In [0]:
#Author : Aimun

# Pharma Manufacturing Project — Historian Bronze → Silver

**Updated fixed version**

This notebook is designed to run **one cell at a time** in Azure Databricks.

Key rules followed:
- Read Bronze as strings first.
- Remove exact duplicates before standardisation.
- Standardise `batch_id` and `equipment_id`.
- Use **tolerant** parsing for mixed timestamps (`try_to_timestamp`).
- Use **tolerant** numeric parsing (`try_cast`).
- Convert Fahrenheit to Celsius.
- Keep abnormal process readings and flag them rather than deleting them.
- Write Silver as Delta to the agreed ADLS path.


In [0]:
# STEP 1 - Import PySpark functions

from pyspark.sql import functions as F

In [0]:
# STEP 2 - Define paths

bronze_path = "abfss://bronze@pharmadatalakeuk.dfs.core.windows.net/historian/historian_readings.csv"

silver_path = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net/historian"

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
# STEP 3 - Read a FRESH Bronze DataFrame
# Everything stays as STRING initially.

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(bronze_path)
)

display(df_raw.limit(10))

In [0]:
# STEP 4 - Record Bronze row count and inspect schema

bronze_count = df_raw.count()

print("Bronze row count:", bronze_count)

df_raw.printSchema()

In [0]:
# STEP 5 - Remove exact duplicate rows BEFORE cleaning

df = df_raw.dropDuplicates()

dedup_count = df.count()

print("Rows before deduplication:", bronze_count)
print("Rows after deduplication :", dedup_count)
print("Exact duplicates removed :", bronze_count - dedup_count)

In [0]:
# STEP 6 - Standardise reading_id and batch_id

df = (
    df
    .withColumn(
        "reading_id",
        F.upper(F.trim(F.col("reading_id")))
    )
    .withColumn(
        "batch_id",
        F.upper(F.trim(F.col("batch_id")))
    )
)

In [0]:
# STEP 7 - Standardise equipment_id
# Example: "Reactor 02" -> "REACTOR-02"

df = df.withColumn(
    "equipment_id",
    F.regexp_replace(
        F.upper(F.trim(F.col("equipment_id"))),
        r"\s+",
        "-"
    )
)

In [0]:
# STEP 8 - Keep raw copies of fields that will be converted

df = (
    df
    .withColumn("reading_timestamp_raw", F.col("reading_timestamp"))
    .withColumn("temperature_raw", F.col("temperature"))
    .withColumn("pressure_raw", F.col("pressure"))
    .withColumn("ph_raw", F.col("ph"))
    .withColumn("flow_rate_raw", F.col("flow_rate"))
    .withColumn("temperature_unit_raw", F.col("temperature_unit"))
)

In [0]:
# STEP 9 - Classify the raw timestamp format
# This is useful for profiling and validation.

df = df.withColumn(
    "timestamp_format",
    F.when(
        F.trim(F.col("reading_timestamp_raw")).rlike(
            r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
        ),
        F.lit("yyyy-MM-dd HH:mm:ss")
    )
    .when(
        F.trim(F.col("reading_timestamp_raw")).rlike(
            r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2}$"
        ),
        F.lit("dd/MM/yyyy HH:mm:ss")
    )
    .when(
        F.trim(F.col("reading_timestamp_raw")).rlike(r"^\d{10}$"),
        F.lit("unix_seconds")
    )
    .otherwise(F.lit("unknown"))
)

display(
    df.groupBy("timestamp_format")
      .count()
      .orderBy("timestamp_format")
)

In [0]:
# STEP 10 - Parse timestamps SAFELY
#
# IMPORTANT:
# try_to_timestamp returns NULL when a format does not match.
# It does NOT throw CANNOT_PARSE_TIMESTAMP.
#
# We try:
# 1. yyyy-MM-dd HH:mm:ss
# 2. dd/MM/yyyy HH:mm:ss
# 3. Unix seconds

timestamp_iso = F.expr(
    "try_to_timestamp(trim(reading_timestamp_raw), 'yyyy-MM-dd HH:mm:ss')"
)

timestamp_dmy = F.expr(
    "try_to_timestamp(trim(reading_timestamp_raw), 'dd/MM/yyyy HH:mm:ss')"
)

timestamp_unix = F.when(
    F.trim(F.col("reading_timestamp_raw")).rlike(r"^\d{10}$"),
    F.from_unixtime(
        F.trim(F.col("reading_timestamp_raw")).cast("long")
    ).cast("timestamp")
)

df = df.withColumn(
    "reading_timestamp",
    F.coalesce(
        timestamp_iso,
        timestamp_dmy,
        timestamp_unix
    )
)

In [0]:
# STEP 11 - Create timestamp quality flags and reading_date

df = (
    df
    .withColumn(
        "timestamp_invalid_flag",
        F.col("reading_timestamp").isNull()
    )
    .withColumn(
        "timestamp_suspicious_flag",
        F.when(
            F.col("reading_timestamp").isNull(),
            F.lit(False)
        ).otherwise(
            F.year(F.col("reading_timestamp")) != 2024
        )
    )
    .withColumn(
        "reading_date",
        F.to_date(F.col("reading_timestamp"))
    )
)

In [0]:
# STEP 12 - Validate timestamp parsing NOW

display(
    df.select(
        "reading_timestamp_raw",
        "timestamp_format",
        "reading_timestamp",
        "reading_date",
        "timestamp_invalid_flag",
        "timestamp_suspicious_flag"
    ).limit(50)
)

print(
    "Invalid timestamps:",
    df.filter(F.col("timestamp_invalid_flag") == True).count()
)

print(
    "Suspicious timestamps:",
    df.filter(F.col("timestamp_suspicious_flag") == True).count()
)

In [0]:
# STEP 13 - Parse sensor values SAFELY using try_cast
#
# try_cast returns NULL for malformed values instead of failing.

df = (
    df
    .withColumn(
        "temperature_parsed",
        F.expr("try_cast(trim(temperature_raw) AS DOUBLE)")
    )
    .withColumn(
        "pressure_parsed",
        F.expr("try_cast(trim(pressure_raw) AS DOUBLE)")
    )
    .withColumn(
        "ph_parsed",
        F.expr("try_cast(trim(ph_raw) AS DOUBLE)")
    )
    .withColumn(
        "flow_rate_parsed",
        F.expr("try_cast(trim(flow_rate_raw) AS DOUBLE)")
    )
)

In [0]:
# STEP 14 - Flag malformed numeric source values
# Blank/null is treated separately as missing data.

df = (
    df
    .withColumn(
        "temperature_invalid_numeric",
        (
            F.col("temperature_raw").isNotNull()
            & (F.trim(F.col("temperature_raw")) != "")
            & F.col("temperature_parsed").isNull()
        )
    )
    .withColumn(
        "pressure_invalid_numeric",
        (
            F.col("pressure_raw").isNotNull()
            & (F.trim(F.col("pressure_raw")) != "")
            & F.col("pressure_parsed").isNull()
        )
    )
    .withColumn(
        "ph_invalid_numeric",
        (
            F.col("ph_raw").isNotNull()
            & (F.trim(F.col("ph_raw")) != "")
            & F.col("ph_parsed").isNull()
        )
    )
    .withColumn(
        "flow_rate_invalid_numeric",
        (
            F.col("flow_rate_raw").isNotNull()
            & (F.trim(F.col("flow_rate_raw")) != "")
            & F.col("flow_rate_parsed").isNull()
        )
    )
)

In [0]:
# STEP 15 - Standardise the temperature unit

df = df.withColumn(
    "temperature_unit_normalised",
    F.upper(F.trim(F.col("temperature_unit_raw")))
)

df = df.withColumn(
    "temperature_unit_invalid_flag",
    (
        F.col("temperature_unit_normalised").isNotNull()
        & (~F.col("temperature_unit_normalised").isin("C", "F"))
    )
)

display(
    df.groupBy("temperature_unit_normalised")
      .count()
      .orderBy("temperature_unit_normalised")
)

In [0]:
# STEP 16 - Convert temperature to Celsius
#
# C stays C
# F is converted to C
# Unknown unit becomes NULL rather than being misinterpreted.

df = df.withColumn(
    "temperature_converted_from_f",
    F.col("temperature_unit_normalised") == "F"
)

df = df.withColumn(
    "temperature",
    F.when(
        F.col("temperature_unit_normalised") == "F",
        (F.col("temperature_parsed") - 32) * 5 / 9
    )
    .when(
        F.col("temperature_unit_normalised") == "C",
        F.col("temperature_parsed")
    )
    .otherwise(F.lit(None).cast("double"))
)

df = df.withColumn(
    "temperature",
    F.round(F.col("temperature"), 2)
)

df = df.withColumn(
    "temperature_unit",
    F.when(
        F.col("temperature_unit_normalised").isin("C", "F"),
        F.lit("C")
    ).otherwise(F.lit(None).cast("string"))
)

# Rename the already parsed numeric fields into their final Silver names.
df = (
    df
    .withColumn("pressure", F.col("pressure_parsed"))
    .withColumn("ph", F.col("ph_parsed"))
    .withColumn("flow_rate", F.col("flow_rate_parsed"))
)

In [0]:
# STEP 17 - Inspect converted sensor values

display(
    df.select(
        "temperature_raw",
        "temperature_unit_raw",
        "temperature",
        "temperature_unit",
        "temperature_converted_from_f",
        "pressure_raw",
        "pressure",
        "ph_raw",
        "ph",
        "flow_rate_raw",
        "flow_rate"
    ).limit(50)
)

In [0]:
# STEP 18 - Create a missing sensor flag
#
# We DO NOT replace missing readings with zero.

df = df.withColumn(
    "missing_sensor_flag",
    F.col("temperature").isNull()
    | F.col("pressure").isNull()
    | F.col("ph").isNull()
    | F.col("flow_rate").isNull()
)

In [0]:
# STEP 19 - Temperature abnormal flag
# Normal operating range: 55 to 75 Celsius

df = df.withColumn(
    "temperature_abnormal",
    F.when(
        F.col("temperature").isNull(),
        F.lit(False)
    ).otherwise(
        ~F.col("temperature").between(55, 75)
    )
)

In [0]:
# STEP 20 - Pressure abnormal flag
# Normal operating range: 1.5 to 4.0 bar

df = df.withColumn(
    "pressure_abnormal",
    F.when(
        F.col("pressure").isNull(),
        F.lit(False)
    ).otherwise(
        ~F.col("pressure").between(1.5, 4.0)
    )
)

In [0]:
# STEP 21 - pH abnormal flag
# Normal operating range: 6.2 to 7.4

df = df.withColumn(
    "ph_abnormal",
    F.when(
        F.col("ph").isNull(),
        F.lit(False)
    ).otherwise(
        ~F.col("ph").between(6.2, 7.4)
    )
)

In [0]:
# STEP 22 - Flow-rate abnormal flag
# Normal operating range: 70 to 130 L/min

df = df.withColumn(
    "flow_rate_abnormal",
    F.when(
        F.col("flow_rate").isNull(),
        F.lit(False)
    ).otherwise(
        ~F.col("flow_rate").between(70, 130)
    )
)

In [0]:
# STEP 23 - Feature engineering:
# Count the number of abnormal sensors on each Historian row.

df = df.withColumn(
    "abnormal_sensor_count",
    F.col("temperature_abnormal").cast("int")
    + F.col("pressure_abnormal").cast("int")
    + F.col("ph_abnormal").cast("int")
    + F.col("flow_rate_abnormal").cast("int")
)

df = df.withColumn(
    "any_abnormal",
    F.col("abnormal_sensor_count") > 0
)

In [0]:
# STEP 24 - Create overall data-quality flags
#
# Data-quality problem != valid process abnormality.
#
# Example:
# 80 C and parsed correctly -> valid data, but process abnormal
# missing temperature       -> data-quality problem

df = df.withColumn(
    "invalid_numeric_flag",
    F.col("temperature_invalid_numeric")
    | F.col("pressure_invalid_numeric")
    | F.col("ph_invalid_numeric")
    | F.col("flow_rate_invalid_numeric")
)

df = df.withColumn(
    "data_quality_issue",
    F.col("timestamp_invalid_flag")
    | F.col("timestamp_suspicious_flag")
    | F.col("missing_sensor_flag")
    | F.col("invalid_numeric_flag")
    | F.col("temperature_unit_invalid_flag")
)

In [0]:
# STEP 25 - Preview process abnormalities

display(
    df.filter(F.col("any_abnormal") == True)
      .select(
          "reading_id",
          "batch_id",
          "equipment_id",
          "reading_timestamp",
          "temperature",
          "pressure",
          "ph",
          "flow_rate",
          "temperature_abnormal",
          "pressure_abnormal",
          "ph_abnormal",
          "flow_rate_abnormal",
          "abnormal_sensor_count",
          "any_abnormal"
      )
      .limit(50)
)

In [0]:
# STEP 26 - Preview data-quality problems

display(
    df.filter(F.col("data_quality_issue") == True)
      .select(
          "reading_id",
          "batch_id",
          "equipment_id",
          "reading_timestamp_raw",
          "reading_timestamp",
          "missing_sensor_flag",
          "timestamp_invalid_flag",
          "timestamp_suspicious_flag",
          "invalid_numeric_flag",
          "temperature_unit_invalid_flag",
          "data_quality_issue"
      )
      .limit(50)
)

In [0]:
# STEP 27 - Build the final Silver DataFrame
#
# Bronze already preserves raw source values, so temporary working
# columns are removed from the final Silver dataset.

df_silver = df.drop(
    "reading_timestamp_raw",
    "temperature_raw",
    "pressure_raw",
    "ph_raw",
    "flow_rate_raw",
    "temperature_unit_raw",
    "temperature_unit_normalised",
    "temperature_parsed",
    "pressure_parsed",
    "ph_parsed",
    "flow_rate_parsed"
)

In [0]:
# Check Historian batch IDs against MES

mes_path = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net/mes"

mes_check = (
    spark.read
    .format("delta")
    .load(mes_path)
)

historian_orphans = df_silver.join(
    mes_check.select("batch_id").distinct(),
    on="batch_id",
    how="left_anti"
)

print("Historian orphan rows:", historian_orphans.count())

display(historian_orphans)

In [0]:
# STEP 28 - Validate the final Silver dataset

silver_count = df_silver.count()

print("========================================")
print("HISTORIAN SILVER VALIDATION")
print("========================================")

print("Bronze rows              :", bronze_count)
print("Silver rows              :", silver_count)
print("Rows removed             :", bronze_count - silver_count)

print(
    "Unique batches           :",
    df_silver.select("batch_id").distinct().count()
)

print(
    "Unique equipment         :",
    df_silver.select("equipment_id").distinct().count()
)

print(
    "Invalid timestamps       :",
    df_silver.filter(F.col("timestamp_invalid_flag") == True).count()
)

print(
    "Suspicious timestamps    :",
    df_silver.filter(F.col("timestamp_suspicious_flag") == True).count()
)

print(
    "Rows with missing sensor :",
    df_silver.filter(F.col("missing_sensor_flag") == True).count()
)

print(
    "Rows with DQ issue       :",
    df_silver.filter(F.col("data_quality_issue") == True).count()
)

print(
    "Rows with abnormalities  :",
    df_silver.filter(F.col("any_abnormal") == True).count()
)

In [0]:
# STEP 29 - Inspect final Silver schema

df_silver.printSchema()

In [0]:
# STEP 30 - Preview final Silver dataset
# This cell deliberately requests the parsed timestamp so we validate
# that mixed timestamp formats are now handled safely.

display(
    df_silver.select(
        "reading_id",
        "batch_id",
        "equipment_id",
        "reading_timestamp",
        "reading_date",
        "timestamp_format",
        "temperature",
        "temperature_unit",
        "pressure",
        "ph",
        "flow_rate",
        "missing_sensor_flag",
        "temperature_abnormal",
        "pressure_abnormal",
        "ph_abnormal",
        "flow_rate_abnormal",
        "abnormal_sensor_count",
        "any_abnormal",
        "data_quality_issue"
    ).limit(100)
)

In [0]:
# STEP 31 - Write Historian Silver as Delta

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path)
)

print("Silver write completed.")

In [0]:
# STEP 32 - Read the Delta data back from Silver

df_check = (
    spark.read
    .format("delta")
    .load(silver_path)
)

print("Rows read back from Silver:", df_check.count())

display(df_check.limit(10))

In [0]:
# STEP 33 - Final completion message

print("========================================")
print("HISTORIAN BRONZE -> SILVER SUCCESSFUL")
print("========================================")
print("Silver path:", silver_path)